# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. Below, we will explore and process the dataset using only `@id` references for all entities.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is a structured object; access fields as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all available record sets and their `@id`s. For each record set, we will also print available fields and columns (by their `@id`).

In [ ]:
# List all record sets with their @id
record_sets = []
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    for record_set in dataset.metadata.recordSet:
        print(f"RecordSet @id: {record_set['@id']}")
        record_sets.append(record_set['@id'])
        # List the fields or columns in each record set if present
        if 'field' in record_set:
            print("  Fields:")
            for field in record_set['field']:
                print(f"    - Field @id: {field['@id']}")
        if 'column' in record_set:
            print("  Columns:")
            for col in record_set['column']:
                print(f"    - Column @id: {col['@id']}")
else:
    print("No record sets available in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's extract all available record sets and load them into pandas DataFrames. We continue to reference entities by `@id` throughout.

In [ ]:
# If record_sets is empty, re-populate from metadata
if not record_sets:
    if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
        record_sets = [r['@id'] for r in dataset.metadata.recordSet]
    else:
        record_sets = []  # No record sets present

# Extract data from each record set (by @id)
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns from the first record set (by @id)
if record_sets:
    print(f"Columns in RecordSet {record_sets[0]}:")
    print(dataframes[record_sets[0]].columns.tolist())
    dataframes[record_sets[0]].head()
else:
    print("No record sets were found to extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, and grouping data. All columns and fields referenced by their `@id`.

We'll demonstrate these steps using one record set, assuming it contains numeric fields and relevant attributes. Be sure to substitute valid `@id`s for the fields as needed.

In [ ]:
# Choose a record set and numeric field by @id
# For demonstration, select the first record set
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    # List fields to identify a numeric one
    print(f"Columns of record set {record_set_id}: {df.columns.tolist()}")
    # Example: Suppose the dataset has a numeric field with @id 'logLikelihood' (replace with actual @id if different)
    numeric_field_id = 'logLikelihood'  # Example field @id, change if needed
    if numeric_field_id in df.columns:
        threshold = -1000  # Example threshold for ordered logistic regression log likelihood
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Example grouping: Suppose grouping by 'ward' (@id of field) if present
        group_field_id = 'ward'
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print(f"Field '{group_field_id}' not found, skipping grouping.")
    else:
        print(f"Numeric field '{numeric_field_id}' not found in columns.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Plotting log likelihood values, coefficients, or other relationships of interest. Entities referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for numeric fields in the selected record set
if record_sets and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If 'ward' field exists, plot mean logLikelihood by ward
    if 'ward' in df.columns:
        plt.figure(figsize=(10, 6))
        sns.barplot(x='ward', y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by ward")
        plt.xlabel("ward")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Required fields for visualization are not present.")

## 6. Conclusion
Summarize key findings from the dataset exploration.

- The Croissant-based dataset is loaded and record sets are referenced by their `@id` throughout.
- Numeric fields such as log likelihood (referenced by `@id`) can be processed and visualized.
- Filtering, normalization, and grouping by key attributes (e.g., `ward` by `@id`) allow insightful exploration.
- Visualizations highlight distribution and group-level differences in regression outputs across surveyed wards.

This notebook demonstrates reproducible FAIR data handling using the `mlcroissant` library, referencing all entities strictly by their `@id`.